# CardioScope-XAI — LSTM Temporal Model
Generates synthetic longitudinal health sequences, trains the LSTM classifier, and extracts temporal embeddings for fusion.

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

from src.data_loader import load_and_clean
from src.temporal_data import build_temporal_dataset, TEMPORAL_FEATURES
from src.lstm_model import train, extract_embeddings, load_encoder

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120
RANDOM_STATE = 42

## 1. Load Clinical Data

In [ ]:
df = load_and_clean('../data/raw/heart_disease_uci.csv')
print(f'Patients: {len(df)}')

## 2. Generate Synthetic Temporal Sequences

In [ ]:
sequences, labels, norm_params = build_temporal_dataset(df, save=True)

print(f'\nSequences shape : {sequences.shape}  (patients × timesteps × features)')
print(f'Labels shape    : {labels.shape}')
print(f'\nNorm params:')
for feat, params in norm_params.items():
    print(f'  {feat:15s}: min={params["min"]:.1f}  max={params["max"]:.1f}')

## 3. Visualise Temporal Trends

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()

high_risk_idx = np.where(labels == 1)[0][:30]
low_risk_idx  = np.where(labels == 0)[0][:30]
timesteps     = np.arange(sequences.shape[1])

for f, (feat, ax) in enumerate(zip(TEMPORAL_FEATURES, axes)):
    for idx in high_risk_idx:
        ax.plot(timesteps, sequences[idx, :, f], color='#F44336', alpha=0.15, lw=1)
    for idx in low_risk_idx:
        ax.plot(timesteps, sequences[idx, :, f], color='#4CAF50', alpha=0.15, lw=1)

    # Mean trends
    ax.plot(timesteps, sequences[high_risk_idx, :, f].mean(axis=0),
            color='#B71C1C', lw=2.5, label='High Risk (mean)')
    ax.plot(timesteps, sequences[low_risk_idx,  :, f].mean(axis=0),
            color='#1B5E20', lw=2.5, label='Low Risk (mean)')

    ax.set_title(feat.replace('_', ' ').title())
    ax.set_xlabel('Month')
    ax.set_ylabel('Normalised Value')
    ax.legend(fontsize=8)

plt.suptitle('Temporal Health Trends — High Risk vs Low Risk', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Train / Validation Split

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    sequences, labels, test_size=0.2, random_state=RANDOM_STATE, stratify=labels
)

print(f'Train: {X_train.shape}  |  Val: {X_val.shape}')
print(f'Train balance — 0: {(y_train==0).sum()}  1: {(y_train==1).sum()}')
print(f'Val   balance — 0: {(y_val==0).sum()}    1: {(y_val==1).sum()}')

## 5. Train LSTM

In [ ]:
full_model, encoder_model, history = train(
    X_train, y_train, X_val, y_val,
    epochs=100,
    batch_size=32,
    learning_rate=0.001,
    log_mlflow=True,
)

print('\n── Validation Metrics ──')
print(f'  Accuracy : {history["val_accuracy"][-1]:.4f}')
print(f'  Recall   : {history["val_recall"][-1]:.4f}')
print(f'  AUC      : {history["val_auc"][-1]:.4f}')

## 6. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

metrics_to_plot = [
    ('loss',     'val_loss',     'Loss'),
    ('accuracy', 'val_accuracy', 'Accuracy'),
    ('recall',   'val_recall',   'Recall'),
]

for ax, (train_key, val_key, title) in zip(axes, metrics_to_plot):
    ax.plot(history[train_key], label='Train', color='steelblue')
    ax.plot(history[val_key],   label='Val',   color='#F44336')
    ax.set_title(title)
    ax.set_xlabel('Epoch')
    ax.legend()

plt.suptitle('LSTM Training History', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Extract Temporal Embeddings

In [ ]:
embeddings = extract_embeddings(encoder_model, sequences)
print(f'Temporal embeddings shape: {embeddings.shape}')  # (n_patients, 16)

# Save embeddings for fusion notebook
import numpy as np
np.save('../data/temporal/lstm_embeddings.npy', embeddings)
print('Saved → data/temporal/lstm_embeddings.npy')

## 8. Embedding Space Visualisation (PCA)

In [ ]:
from sklearn.decomposition import PCA

pca   = PCA(n_components=2, random_state=RANDOM_STATE)
emb2d = pca.fit_transform(embeddings)

fig, ax = plt.subplots(figsize=(8, 6))
for label, color, name in [(0, '#4CAF50', 'No Disease'), (1, '#F44336', 'Disease')]:
    mask = labels == label
    ax.scatter(emb2d[mask, 0], emb2d[mask, 1],
               c=color, label=name, alpha=0.7, s=40, edgecolors='white', lw=0.5)

ax.set_title('LSTM Temporal Embeddings — PCA (2D)', fontsize=13, fontweight='bold')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
ax.legend()
plt.tight_layout()
plt.show()

## Summary

| Item | Detail |
|---|---|
| Input shape | (n_patients, 12, 4) |
| Features | Systolic BP, Heart Rate, Cholesterol, ST Depression |
| Architecture | LSTM(64) → LSTM(32) → Dense(16) → Dense(1) |
| Embedding dim | 16 (used in fusion layer) |
| Saved | `models/lstm_model.keras`, `models/lstm_encoder.keras` |
| Embeddings | `data/temporal/lstm_embeddings.npy` |